# 10 · E5 semantic embedding

**Tujuan:** Menjalankan intfloat/multilingual-e5-base dengan tahapan tokenisasi model, embedding, similarity, dan penyimpanan. Model pretrained diunduh sekali; tidak ada Google Books API.

**Alur:** input → proses yang terlihat → pengecekan → output. Semua label kompetensi lama hanya dipakai sebagai referensi audit, bukan kebenaran relevansi.

## 0. Setup dan validasi sumber data

In [1]:
from pathlib import Path
import os, sys, json, time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
ROOT = Path.cwd()
if not (ROOT/'data'/'dataset_buku_2000.csv').exists(): ROOT=ROOT.parent
assert (ROOT/'data'/'dataset_buku_2000.csv').exists(), f'Folder proyek tidak ditemukan dari {Path.cwd()}'
sys.path.insert(0,str(ROOT))
from src import engine
DATA, OUT, FIG, MODELS = [ROOT/x for x in ('data','outputs','figures','models')]
for folder in (OUT,FIG,MODELS): folder.mkdir(exist_ok=True)
books, comps = engine.load_data()
print('Dataset:',len(books),'buku;',len(comps),'kompetensi')
assert books.book_id.is_unique and comps.competency_id.is_unique

Dataset: 973 buku; 67 kompetensi


## 1. Periksa dependensi dan input

Pengunduhan bobot model memerlukan internet pada penggunaan pertama.

In [2]:
clean=pd.read_csv(OUT/'dataset_buku_bersih.csv',dtype=str).fillna('')
profiles=pd.read_csv(OUT/'profil_kompetensi_query.csv',dtype=str).fillna('')
assert len(clean)==len(books) and len(profiles)==len(comps)
books['teks_buku']=clean.set_index('book_id').reindex(books.book_id)['teks_model'].fillna('').to_numpy() if 'teks_model' in clean.columns else books.teks_buku
comps['teks_query']=profiles.set_index('competency_id').reindex(comps.competency_id)['teks_query'].fillna('').to_numpy() if 'teks_query' in profiles.columns else comps.teks_query
assert books.teks_buku.str.len().gt(0).all()
print('Contoh buku:',books[['judul','teks_buku']].head(2).to_string(index=False,max_colwidth=110))
print('Contoh kompetensi:',comps[['competency_name','teks_query']].head(2).to_string(index=False,max_colwidth=110))
import importlib.util
assert importlib.util.find_spec('sentence_transformers') is not None, 'Install requirements-semantic.txt lalu jalankan ulang notebook'

Contoh buku:                                judul                                                                                                      teks_buku
            Business Growth Strategy Business Growth Strategy Business Growth Strategy Introduction I wrote this book, “Business Growth Strategy...
Mastering Business Growth Strategies Mastering Business Growth Strategies Mastering Business Growth Strategies Mastering Business Growth Strateg...
Contoh kompetensi:      competency_name                                                                          teks_query
Business Development       Business Development business growth; business strategy; market opportunities
Marketing Management Marketing Management marketing strategy; marketing analytics; customer segmentation


## 2. Inisialisasi model dan catat dimensi

Tidak menyimpan angka performa jika unduhan atau inferensi belum berhasil.

In [3]:
from sentence_transformers import SentenceTransformer
MODEL_ID='intfloat/multilingual-e5-base'
t0=time.perf_counter()
model=SentenceTransformer(MODEL_ID)
print('Nama model:',MODEL_ID,'dimensi embedding:',model.get_sentence_embedding_dimension())

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Nama model: intfloat/multilingual-e5-base dimensi embedding: 768


C:\Users\AUREL ELVIOLITA\AppData\Local\Temp\ipykernel_25628\1439645922.py:5: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print('Nama model:',MODEL_ID,'dimensi embedding:',model.get_sentence_embedding_dimension())


## 3. Buat embedding buku dalam batch

Embedding katalog dapat disimpan dan dipakai ulang ketika kompetensinya berubah.

In [4]:
texts=["passage: " + str(x) for x in books.teks_buku]
book_vec=model.encode(texts,batch_size=8,show_progress_bar=True,normalize_embeddings=True)
print('Bentuk embedding buku:',book_vec.shape)
np.save(MODELS/'e5_books.npy',book_vec)

Batches:   0%|          | 0/122 [00:00<?, ?it/s]

Bentuk embedding buku: (973, 768)


## 4. Buat embedding kompetensi, hitung seluruh skor

E5 menggunakan prefix query:/passage: sesuai mekanisme retrieval-nya.

In [5]:
queries=["query: " + str(x) for x in comps.teks_query]
query_vec=model.encode(queries,batch_size=8,show_progress_bar=True,normalize_embeddings=True)
scores=query_vec@book_vec.T
print('Skor matrix:',scores.shape,'range:',scores.min(),scores.max())
assert scores.shape==(len(comps),len(books))

Batches:   0%|          | 0/9 [00:00<?, ?it/s]

Skor matrix: (67, 973) range: 0.682092 0.8850932


## 5. Audit ranking dan simpan artefak

Tidak disimulasikan bila model belum dijalankan.

In [6]:
elapsed=time.perf_counter()-t0
pairs,top=engine.save_all_pairs(scores,books,comps,'e5')
pd.DataFrame([{'model':'e5','waktu_total_detik':elapsed,'jumlah_pasangan':len(pairs)}]).to_csv(OUT/'runtime_e5.csv',index=False)
display(top.loc[top.competency_id==comps.competency_id.iloc[0],['ranking','judul','skor_asli']].head(10))
assert len(pairs)==len(books)*len(comps)
print('SELESAI: ',len(pairs),'pasangan tercatat')

,ranking,judul,skor_asli
0,1,Mastering Business Growth Strategies,0.855498
1,2,Business Development For Dummies,0.851218
2,3,Business Growth Strategy,0.841250
3,4,AI-Powered Business Growth Strategies,0.838626
4,5,Mastering Business Strategy,0.837123
5,6,Emerging Business Opportunities in Africa: Mar...,0.830249
6,7,Business Model Renewal: How to Grow and Prospe...,0.828573
7,8,"Innovation, Product Development and Commercial...",0.828449
8,9,The Purpose of Business. Where Value meets Str...,0.824853
9,10,Developing Strategies for Competitive Advantage,0.822077


SELESAI:  65191 pasangan tercatat


## Kesimpulan tahap dan serah-terima

Periksa tabel/grafik yang tercetak, simpan artefak hasil, lalu lanjut ke notebook berikutnya. **Kesamaan kata, similarity dan percentile bukan ground truth relevansi.** Jangan mengklaim metrik akurasi tanpa penilaian independen.